# 6. 분류를 위한 fine-tuning

* 지금까지 LLM 구조 구현 + 사전 훈련 (사전 훈련된 가중치를 직접 만든 모델에 로드)
* 이 장에서 다룰 예제: 텍스트 메시지를 "스팸" or "스팸 아님"으로 분류

## 6.1. 여러 가지 미세 튜닝 (fine-tuning) 방법

* 지시 미세 튜닝 (instruction fine-tuning): 자연어 프롬프트로 설명된 작업을 이해하고 실행하는 능력 향상
    - 분류 미세 튜닝 모델에 비해 광범위한 작업 수행 가능
* 분류 미세 튜닝 (classification fine-tuning): 일련의 클래스 레이블을 인식하도록 훈련
    - 분류 미세 튜닝 모델이 훈련 과정에서 만난 클래스만 예측 



In [ ]:
import torch
import torch.nn as nn
from torch.nn import LayerNorm

## 6.2. 데이터셋 준비

#### 데이터셋 다운로드 + 압축 풀기
* sms_spam_collection 폴더 안에 탭으로 구분된 텍스트 파일 SMSSpamCollection.tsv로 저장

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path}가 이미 있어 다운로드 및 압축 해제를 건너뜁니다.")
        return

    # 파일 다운로드
    with urllib.request.urlopen(url) as response:  
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # 파일 압축 해제
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"  
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}에 저장되었습니다.")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [ ]:
import pandas as pd
df = pd.read_csv(
    data_file_path, sep="\t", header=None, names=["Label", "Text"]
)
print(df)

#### 클래스 레이블 분포

In [ ]:
print(df["Label"].value_counts())

#### 언더샘플링으로 클래스 불균형 해결

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

#### 문자열 클래스 레이블 "ham"과 "spam"을 정수 클래스 레이블 0, 1로 각각 변경

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

#### 데이터셋 세 부분 분할 (70%: train, 10%: validate, 10% test)

In [ ]:
def random_split(df, train_frac, validation_frac):
    
    # 전체 데이터프레임 섞기
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    
    # 분할할 인덱스 계산
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # 데이터프레임 분할 ㄴ
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

# 테스트 크기는 나머지에 해당하는 0.2
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

print("train:", train_df.shape)
print("validation:", validation_df.shape)
print("test:", test_df.shape)
print("\ntrain label counts:")
print(train_df["Label"].value_counts())
print("\nvalidation label counts:")
print(validation_df["Label"].value_counts())
print("\ntest label counts:")
print(test_df["Label"].value_counts())

#### 재사용을 위한 데이터셋 저장

In [ ]:
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

## 6.3. 데이터 로더 만들기

* 이전: 슬라이딩 윈도우로 균일한 크기의 text chunk 생성 
    - 효율적인 모델 훈련을 위해 이를 배치로 묶었음
    - 각 chunk는 개별 훈련 샘플
* 이번: 스팸 데이터셋의 길이가 다양하므로 text chunk처럼 메시지를 배치로 묶으려면 두 가지 방법 중 하나를 선택해야 함
    1. 데이터셋이나 배치에 있는 가장 짧은 길이의 메시지에 맞춰 모든 메시지를 잘라냄
        - 계산 비용이 저렴함
        - 가장 짧은 메시지는 평균이나 가장 긴 메시지에 비해 길이가 훨씬 짧아서 정보 손실이 많음
        - 결과적으로 모델 성능이 낮아지기 쉬움
    2. 데이터셋이나 배치에 있는 가장 긴 길이의 메시지에 맞춰 모든 메시지에 패딩 추가
        - 메시지 정보 손실은 아예 없음
        - 계산 비용이 큼
        - 여기서는 이 방식으로 진행

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

#### 패딩 토큰: <|endoftext|>
* <|endoftext|> 토큰 대신 이에 해당하는 토큰 ID 50256 추가

#### pytorch dataset class 준비 (SpamDataset 클래스)
* CSV 파일에서 데이터 로드
* tiktoken의 GPT-2 토크나이저로 텍스트 토큰화
* 가장 긴 시퀀스나 사전에 정의된 최대 길이에 맞춰 동일한 길이가 되도록 시퀀스를 자르고 패딩 추가
    - 텐서를 동일한 길이로 만듦

In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id

        # 텍스트 토큰화
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            
            # max_length보다 긴 시퀀스 자르기
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 가장 긴 시퀀스에 맞춰 패딩 추가
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long),
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

#### 훈련 데이터 로더에서 배치 만들기

In [ ]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

In [ ]:
print(train_dataset.max_length)

#### 120이면 일반적인 텍스트 메시지 길이
* 이 모델은 문맥 길이 한도에 해당하는 1024개 토큰까지 처리 가능
* 데이터셋에 더 긴 텍스트가 있다면 모델이 지원하는 입력(문맥) 길이를 초과하지 않도록 앞의 코드에서 train dataset을 만들 때 max_length=1024로 지정하기

#### *중요한 점
* 훈련 세트의 가장 긴 시퀀스 길이에 맞춰 검증 세트와 테스트 세트에 패딩 추가
* 가장 긴 훈련 샘플의 길이보다 긴 모든 검증 세트 샘플과 테스트 세트 샘플은 SpamDataset의 encoded_text[:self.max_length]로 잘라야 함
* 이렇게 자르는 건 선택사항
* 검증/테스트 샘플에 1024개 토큰을 초과하는 시퀀스가 없으면 max_length=None으로 지정 가능

In [ ]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

#### 위 데이터셋을 입력으로 사용한다면
* 텍스트 데이터 사용 때처럼 데이터 로더를 만들 수 있음
* 이 경우 타겟은 텍스트 다음 토큰이 아닌 클래스 레이블
* 배치 크기를 8로 했다면 각 배치는 길이가 120인 8개의 훈련 샘플과 각 샘플에 해당하는 클래스 레이블로 구성됨

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
	dataset=train_dataset,
	batch_size=batch_size,
	shuffle=True,
	num_workers=num_workers,
	drop_last=True,
)

val_loader = DataLoader(
	dataset=val_dataset,
	batch_size=batch_size,
	num_workers=num_workers,
	drop_last=False,
)

test_loader = DataLoader(
	dataset=test_dataset,
	batch_size=batch_size,
	num_workers=num_workers,
	drop_last=False,
)

#### 텍스트 메시지와 레이블 로드
* 배치 크기 8인 훈련, 검증, 테스트 세트 데이터 로더 만들기

* 입력 배치: 8개 샘플 + 각 샘플은 120개 토큰
* 레이블 배치: 8개의 훈련 샘플에 해당하는 클래스 레이블 저장

In [ ]:
for input_batch, target_batch in train_loader:
    pass
print("입력 배치 차원: ", input_batch.shape)
print("레이블 배치 차원: ", target_batch.shape)

In [ ]:
print(f"{len(train_loader)}개 훈련 배치")
print(f"{len(val_loader)}개 검증 배치")
print(f"{len(test_loader)}개 테스트 배치")

## 6.4. 사전 훈련된 가중치로 모델 초기화

#### 사전 훈련된 모델 초기화

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257, # 어휘사전 크기
    "context_length": 1024, # 문맥 길이
    "drop_rate": 0.0, # 드롭아웃 비율
    "qkv_bias": True, # Query-Key-Value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

print(BASE_CONFIG)

#### 사전 훈련된 가중치 다운로드 + GPT 모델로 로드

In [ ]:
import torch
import torch.nn as nn
from torch.nn import LayerNorm

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        # 원하는 출력 차원에 맞도록 투영 차원을 낮춤
        self.head_dim = d_out // num_heads

        # Query, Key, Value 선형 변환
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 여러 head의 출력을 다시 섞어주는 출력 projection
        # Linear층을 사용해 헤드의 출력을 결합
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # causal mask
        # 위쪽 삼각행렬을 1로 만들어서 미래 토큰을 보지 못하게 함
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        # 텐서 크기: (b, num_tokens, d_out)
        # x: (batch_size, num_tokens, d_in)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads 차원을 추가함으로써 암묵적으로 행렬 분할
        # 그런 다음 마지막 차원을 num_heads에 맞춰 채움
        # keys, queries, values:
        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # head 차원을 앞으로 이동
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # attention score 계산
        # queries: (b, num_heads, num_tokens, head_dim)
        # keys.transpose(2, 3): (b, num_heads, head_dim, num_tokens)
        # 결과: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # 현재 토큰 수에 맞게 mask 자르기
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 미래 토큰 위치를 -inf로 채움
        # 마스크를 사용해 어텐션 점수를 채움
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # scaled dot-product attention
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # attention weight와 value 곱하기
        # attn_weights: (b, num_heads, num_tokens, num_tokens)
        # values:       (b, num_heads, num_tokens, head_dim)
        # 결과:          (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # 다시 head 차원을 뒤로 보냄
        # 텐서 크기: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # 여러 head를 하나로 결합
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        # 마지막 출력 projection (선형 투영)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x # 어텐션 블록을 위한 숏컷 연결
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        shortcut = x # 피드포워드 신경망을 위한 숏컷 연결
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        return x

In [ ]:
import torch.nn as nn

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        
        # 장치 설정을 통해 입력 데이터가 어디에 있는지에 따라 모델을 CPU나 GPU에서 훈련 가능
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"크기가 다릅니다. left: {left.shape}, right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params): # 위치 임베딩과 토큰 임베딩의 가중치를 params의 값으로 설정
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])): # 모델의 트랜스포커 블록 순회
        q_w, k_w, v_w = np.split( # np.split 함수로 어텐션 가중치와 편향 가중치를 쿼리, 키, 값 세 부분으로 나눔
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"]
        )
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"]
        )

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    # 오픈 AI의 원본 GPT-2 모델은 토큰 임베딩의 가중치를 출력 층에 재사용하여 전체 파라미터 개수를 절감하는 가중치 묶기를 사용함
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
# gpt_download.py에서 download_and_load_gpt2 함수 임포트
# ch.05의 GPTModel 클래스와 load_weights_into_gpt 함수 재사용하려 사전 훈련된 가중치 다운로드 -> GPT 모델로 로드 
from gpt_download import download_and_load_gpt2

model_size = CHOOSE_MODEL.split(" ")[-1].strip("()")
settings, params = download_and_load_gpt2(
    model_size=model_size, models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
def generate_text_simple(model, idx, # idx는 현재 문맥이 담긴 [batch_size, num_tokens] 크기의 인덱스 배열
                         max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        logits = logits[:, -1, :] # 마지막 타임 스텝만 사용하므로 [batch_size, vocab_size]가 [batch_size, num_tokens, vocab_size]에서 [batch_size, vocab_size]로 축소됨
        probas = torch.softmax(logits, dim=-1) # probas는 [batch_size, vocab_size] 크기의 확률 분포
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # idx_next는 [batch_size, 1] 크기의 텐서로, 각 샘플에 대해 가장 높은 확률을 가진 다음 토큰의 인덱스
        idx = torch.cat((idx, idx_next), dim=1) # 선택한 인덱스를 현재 시퀀스에 추가하므로 idx의 크기는 [batch_size, num_tokens + 1]이 됨
    
    return idx

In [ ]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # unsqueeze(0)으로 배치 차원을 추가
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원 제거
    return tokenizer.decode(flat.tolist())

In [ ]:
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

#### 모델을 스팸 분류기로 fine-tuning 하기 전에 명령 지시 프롬프트로 스팸 메시지를 분류할 수 있는가?
* 못함!

In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':\n"
    "'You are a winner you have been specially "
    "selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"],
)

print(token_ids_to_text(token_ids, tokenizer))

## 6.5. 분류 헤드 추가
* 은닉 표현을 어휘사전의 50,257개 토큰에 대한 로짓으로 매핑하는 원래 출력 층을 2개의 클래스(0/1)로 매핑하는 작은 층으로 바꿔야 함
    - 이진분류 작업이라 기술적으로 하나의 출력 노드만 사용 가능함. 그러나 이를 위해 손실 함수를 바꿔야 함
    - 일반적인 방식: 출력 노드 개수를 클래스 개수에 맞춤

#### GPTModel은 임베딩 층 + 12개 트랜스포머 블록 + LayerNorm층 + out_head(출력층)

In [ ]:
print(model)

#### 모델 동결 (freeze)
* 모든 층이 훈련되지 않도록 함

In [ ]:
for param in model.parameters():
    param.requires_grad = False

#### out_head(출력층)을 fine-tuning할 새로운 출력층으로 바꾸기
* 분류층 추가
* 50,257차원(어휘사전 크기)으로 입력을 매핑하던 출력층(model.out_head) 바꾸기
* 새로운 model.out_head층의 requires_grad 속성은 기본적으로 True
    - 방금 추가한 출력층만 훈련 과정에서 업데이트됨
    - 물론 출력 층을 훈련하는 것만으로도 충분하지만, 추가로 다른 층을 fine-tuning 하면 모델의 예측 성능이 향상됨

In [ ]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
    # 코드 재사용성을 위해 gpt2-small (124M) 모델의 임베딩 차원인 768 대신 BASE_CONFIG["emb_dim"] 변수 사용
    in_features=BASE_CONFIG["emb_dim"],
    out_features=num_classes
)

#### 마지막 Transformer 블록 + LayerNorm 모듈 훈련 가능하게 설정

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

#### 입력을 4개의 토큰으로 구성된 텐서로 인코딩하는 것을 볼 수 있음

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("입력:", inputs)
print("입력 차원:", inputs.shape)  # 크기: (배치 크기, 토큰 수)

#### 인코딩된 토큰 ID를 모델에 전달
* outputs.shape 결과 50,257이 아닌 2가 되는 걸 볼 수 있음

In [ ]:
with torch.no_grad():
    outputs = model(inputs)
print("출력:\n", outputs)
print("출력 텐서:\n", outputs.shape)

#### 4개의 출력 행을 모두 fine-tuning 할 필요 x
* 하나의 출력 토큰에만 초점을 맞출 수 있음
* 다음 그림에서는 마지막 출력 토큰에 해당하는 마지막 행에 초점을 맞추기

In [ ]:
print("마지막 출력 토큰:", outputs[:, -1, :])

#### 이 값을 클래스 레이블 예측으로 바꿔야 함

#### 근데 왜 마지막 출력 토큰에만 관심을 두는 것일까?

* GPT 모델에서 사용하는 causal attention mask
    - 마스크: 토큰이 현재 위치와 이전 위치에만 초점을 맞추도록 제한
    - 각 토큰은 자기 자신과 이전 토큰에만 영향을 받음

* 이러한 causal attention mask 설정으로 인해 sequence에 있는 마지막 토큰이 가장 많은 정보 제공
    - 이 토큰이 이전에 있는 모든 토큰의 데이터에 접근 가능함
    - 따라서 스팸 분류 작업을 위한 fine-tuning 과정에서 마지막 토큰에 초점을 맞춤

## 6.6. 분류 손실 + 정확도 계산
* 모델 출력을 클래스 레이블 예측으로 바꾸는 방법
    - 2차원 출력을 softmax 함수를 통해 확률로 바꾸고 argmax 함수로 가장 높은 확률의 위치 반환

In [ ]:
print("마지막 출력 토큰: ", outputs[:, -1, :])

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("클래스 레이블:", label.item()) # 스팸으로 예측

#### 꼭 softmax를 써야 하나
* 사실 가장 큰 출력 = 가장 큰 확률 점수
* 이 경우에는 softmax를 굳이 쓰진 않아도 됨

In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("클래스 레이블: ", label.item())

#### 분류 정확도 계산을 위한 calc_accuracy_loader 함수

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :] # 마지막 출력 토큰의 로짓
                predicted_labels = torch.argmax(logits, dim=-1)
            
            num_examples += predicted_labels.shape[0]
            correct_predictions += (
                (predicted_labels == target_batch).sum().item()
            )
        else:
            break
    
    return correct_predictions / num_examples

#### 10개의 배치로 데이터셋의 분류 정확도 추정

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(
    train_loader, model, device, num_batches=10
)
val_accuracy = calc_accuracy_loader(
    val_loader, model, device, num_batches=10
)
test_accuracy = calc_accuracy_loader(
    test_loader, model, device, num_batches=10
)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

#### fine-tuning 전 훈련 과정에서 최적화할 loss 정의
* 분류 정확도: 미분 불가능
* cross-entropy loss를 surrogate loss function으로 사용
* calc_loss_batch: 전체 토큰인 model(input_batch)가 아니라 마지막 토큰인 model(input_batch)[:, -1, :]만 최적화

In [ ]:
# 데이터 로더에서 반환한 배치 하나에 대한 손실 계산
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :] # 마지막 출력 토큰의 로짓
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

#### 데이터 로더에 있는 전체 배치에 대한 손실 계산 (calc_loss_loader 함수)

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    model.eval()
    total_loss = 0.0

    if len(data_loader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(data_loader)
    else:  # 배치 개수가 데이터 로더에 있는 배치를 초과하지 않도록 함
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches

#### 훈련 정확도 계산할 떄와 비슷하게 다른 데이터셋의 초기 손실 계산

In [ ]:
with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"훈련 손실: {train_loss:.3f}")
print(f"검증 손실: {val_loss:.3f}")
print(f"테스트 손실: {test_loss:.3f}")

## 6.7. 지도 학습 데이터로 모델 fine-tuning

* fine-tuning 훈련 루프
    - 사전 훈련과 다른 점: 모델 평가를 위해 샘플 텍스트 생성 대신 분류 정확도 계산

#### train_classifier_simple
* 사전훈련에 사용한 train_model_simple 함수와 비슷함
    - 차이점 1. 토큰 개수 대신 지금까지 처리한 훈련 샘플 개수 (examples_seen) 세기
    - 차이점 2. 샘플 텍스트 생성 대신 에포크가 끝날 때마다 정확도 계산

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

In [ ]:
def train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs, eval_freq, eval_iter):
    # 손실과 지금까지 처리한 샘플 수를 추적하기 위해 리스트 초기화
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # 메인 훈련 루프 시작
    for epoch in range(num_epochs):
        # 모델 훈련 모드
        model.train()

        for input_batch, target_batch in train_loader:
            
            # 이전 배치 반복에서 얻은 loss gradient 초기화
            optimizer.zero_grad()
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )

            # loss gradient 계산 (backpropagation)
            loss.backward()
            
            # loss gradient를 사용하여 모델 가중치 업데이트
            optimizer.step()
            
            # 토큰이 아닌 샘플 개수 추적
            examples_seen += input_batch.shape[0]
            global_step += 1

            # 추가적인 평가 단계
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"에포크 {epoch+1} (Step {global_step:06d}): "
                      f"훈련 손실 {train_loss:.3f}, "
                      f"검증 손실 {val_loss:.3f}"
                )

        # 각 에포크 후 정확도 계산
        train_accuracy = calc_accuracy_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_accuracy = calc_accuracy_loader(
            val_loader, model, device, num_batches=eval_iter
        )

        print(f"훈련 정확도: {train_accuracy*100:.2f}% | ", end="")
        print(f"검증 정확도: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

#### 옵티마이저 초기화 + train_classifier_simple 함수로 훈련 (epoch = 5)

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5

train_losses, val_losses, train_accs, val_accs, examples_seen = \
    train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=50,
        eval_iter=5
    )

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분.")

#### Matplotlib을 활용한 train, validation dataset에 대한 loss function


In [ ]:
import matplotlib.pyplot as plt


def plot_values(
        epochs_seen, examples_seen, train_values, val_values,
        label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 각 에포크에 대한 훈련과 검증 손실 그래프
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(
        epochs_seen, val_values, linestyle="-.",
        label=f"Validation {label}"
    )

    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # 처리한 샘플 개수를 위해 두 번째 x축 만들기
    ax2 = ax1.twiny()

    # 눈금을 정렬하기 위한 투명한 그래프
    ax2.plot(examples_seen, train_values, alpha=0)

    ax2.set_xlabel("Examples seen")

    # 레이아웃의 간격을 맞춤
    fig.tight_layout()

    plt.savefig(f"{label}-plot.pdf")
    plt.show()


epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

#### plot_values 함수로 분류 정확도 그리기
* epoch 4, 5 이후에 비교적 높은 train/validation accuracy를 달성함
* train_classifier_simple 함수를 사용할 때 eval_iter=5로 지정
    - 효율성을 위해 훈련 과정에서 훈련/검증 성능을 5개의 배치만으로 추정한다는 뜻

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(
    epochs_tensor, examples_seen_tensor, train_accs, val_accs,
    label="accuracy"
)

#### 전체 데이터셋에서 train/validation/test set에 대한 성능 계산
* 이번에는 eval_iter 매개변수 지정 x

* drop_rate나 옵티마이저의 weight_decay 매개변수를 증가시키는 등 모델 설정으로 검증/테스트 세트 간 성능 차이를 줄일 수 있음

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

## 6.8. LLM을 스팸 분류기로 사용

#### 모델을 사용하여 새로운 텍스트 분류 (classify_review 함수)
* SpamDataset에서 사용한 것과 비슷한 데이터 전처리 수행
* 텍스트 -> 토큰 ID로 변환한 후 비슷하게 모델을 통해 정수 클래스 레이블 예측

In [ ]:
def classify_review(
        text, model, tokenizer, device, max_length=None,
        pad_token_id=50256):
    model.eval()

    # 모델을 위해 입력 준비
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    # 시퀀스가 너무 길면 자르기
    input_ids = input_ids[:min(
        max_length, supported_context_length
    )]

    assert max_length is not None, (
        "max_length가 지정되지 않았습니다. 모델의 최대 문맥 길이를 사용하려면 "
        "max_length=model.pos_emb.weight.shape[0]로 지정하세요."
    )

    assert max_length <= supported_context_length, (
        f"max_length({max_length})가 모델이 지원하는 문맥 길이({supported_context_length})를 초과"
        "했습니다."
    )

    # 또는 max_length=None인 경우를 안정적으로 처리하는 방법
    # max_len = min(max_length, supported_context_length) if max_length else supported_context_length

    # input_ids = input_ids[:max_len]

    # 가장 긴 시퀀스에 맞춰 시퀀스에 패딩 추가
    input_ids += [pad_token_id] * (max_length - len(input_ids))

    input_tensor = torch.tensor(
        input_ids, device=device
    ).unsqueeze(0)  # 배치 차원 추가

    # 모델 추론을 위해 그레이디언트 추적을 끔
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # 마지막 출력 토큰의 로짓
    predicted_label = torch.argmax(logits, dim=-1).item()

    # 분류된 결과 반환
    return "스팸" if predicted_label == 1 else "스팸 아님"

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? let me know!"
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

print(classify_review(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

#### 모델 재사용을 위해 저장 (재훈련 필요 x)

In [ ]:
torch.save(model.state_dict(), "review_classifier.pth")

#### 모델 로드

In [ ]:
model_state_dict = torch.load("review_classifier.pth", map_location=device)
model.load_state_dict(model_state_dict)